# B05 Type 1 LLM Pipeline Evaluation

Notebook này dùng để đánh giá chất lượng pipeline Type 1 hiện tại trên input Kaggle/local.

Ràng buộc của notebook:
- Chỉ chạy Logic Type 1.
- Chạy LLM bằng Transformers thông qua `exact.llm_client.build_json_client_from_settings`.
- Gọi codebase hiện tại qua `exact.*`, không inline lại pipeline/solver/parser.
- LLM-only: Type 1 không có heuristic fallback.
- Premise translation đi qua KB cache của codebase; YNU query dùng predicate dictionary từ KB.
- Pipeline dùng k-sampling + symbolic consistency voting theo `EXACT_TYPE1_TRANSLATION_SAMPLES`.
- Nếu symbolic vote trả `Unknown`, codebase sẽ gọi CoT fallback và notebook sẽ log dấu vết đó.
- Nếu LLM/pipeline lỗi, lỗi được ghi vào prediction/report để đánh giá stability; notebook không thay thế bằng heuristic answer.


In [ ]:
from __future__ import annotations

import json
import os
import sys
import time
import traceback
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates.extend([
        Path('/kaggle/working/Exact2026'),
        Path('/kaggle/working'),
    ])
    for candidate in candidates:
        if (candidate / 'src' / 'exact').exists():
            return candidate
    raise FileNotFoundError('Cannot find project root containing src/exact')


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SRC_DIR      =', SRC_DIR)


In [ ]:
# Evaluation knobs. Override with environment variables on Kaggle when needed.
MODEL_NAME = os.getenv('EXACT_LLM_MODEL', 'Qwen/Qwen2.5-1.5B-Instruct')
MAX_NEW_TOKENS = int(os.getenv('EXACT_MAX_NEW_TOKENS', '4096'))
LLM_TEMPERATURE = float(os.getenv('EXACT_LLM_TEMPERATURE', '0.0'))
LLM_TOP_P = float(os.getenv('EXACT_LLM_TOP_P', '1.0'))
TYPE1_TRANSLATION_SAMPLES = int(os.getenv('EXACT_TYPE1_TRANSLATION_SAMPLES', '3'))
TYPE1_SAMPLING_TEMPERATURE = float(os.getenv('EXACT_TYPE1_SAMPLING_TEMPERATURE', '0.7'))
LIMIT_TEXT = os.getenv('EXACT_LIMIT', '').strip()
LIMIT = int(LIMIT_TEXT) if LIMIT_TEXT else None
PROGRESS_EVERY = int(os.getenv('EXACT_PROGRESS_EVERY', '25'))
LOG_EVERY = int(os.getenv('EXACT_LOG_EVERY', '1'))
LOG_TRACEBACK = os.getenv('EXACT_LOG_TRACEBACK', '0') == '1'
CONTINUE_ON_ERROR = os.getenv('EXACT_CONTINUE_ON_ERROR', '1') != '0'
CLEAR_KB_CACHE = os.getenv('EXACT_CLEAR_KB_CACHE', '1') != '0'

default_output_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'outputs' / 'logic'
OUTPUT_PATH = Path(os.getenv('EXACT_LOGIC_OUTPUT', str(default_output_dir / 'type1_llm_predictions.json')))
REPORT_PATH = Path(os.getenv('EXACT_LOGIC_REPORT', str(default_output_dir / 'type1_llm_eval_report.json')))

print({
    'model': MODEL_NAME,
    'max_new_tokens': MAX_NEW_TOKENS,
    'llm_temperature': LLM_TEMPERATURE,
    'llm_top_p': LLM_TOP_P,
    'type1_translation_samples': TYPE1_TRANSLATION_SAMPLES,
    'type1_sampling_temperature': TYPE1_SAMPLING_TEMPERATURE,
    'limit': LIMIT,
    'log_every': LOG_EVERY,
    'log_traceback': LOG_TRACEBACK,
    'continue_on_error': CONTINUE_ON_ERROR,
    'clear_kb_cache': CLEAR_KB_CACHE,
    'output': str(OUTPUT_PATH),
    'report': str(REPORT_PATH),
})


In [ ]:
def resolve_logic_input() -> Path:
    explicit = os.getenv('EXACT_LOGIC_INPUT', '').strip()
    if explicit:
        path = Path(explicit)
        if path.exists():
            return path
        raise FileNotFoundError(f'EXACT_LOGIC_INPUT does not exist: {path}')

    search_roots = [
        Path('/kaggle/input'),
        PROJECT_ROOT / 'src' / 'exact' / 'datasets' / 'exact',
        PROJECT_ROOT,
    ]
    preferred_names = [
        'Logic_Based_Educational_Queries_inference.json',
        'Logic_Based_Educational_Queries.json',
    ]
    for root in search_roots:
        if not root.exists():
            continue
        for name in preferred_names:
            matches = sorted(root.rglob(name)) if root.is_dir() else []
            if matches:
                return matches[0]

    raise FileNotFoundError('Cannot locate Type 1 logic input JSON. Set EXACT_LOGIC_INPUT.')


INPUT_PATH = resolve_logic_input()
print('INPUT_PATH =', INPUT_PATH)


In [ ]:
from exact.common.schemas import PredictionRequest, TaskType, to_official_response
from exact.config import Settings
from exact.datasets.loader import load_logic_dataset
from exact.llm_client import build_json_client_from_settings
from exact.logic.kb import clear_kb_cache
from exact.logic.pipeline import run_type1_pipeline
from exact.router.task_router import TaskRouter


def load_type1_examples(path: Path) -> list[dict]:
    payload = json.loads(path.read_text(encoding='utf-8'))

    # Labeled grouped training/dev style: records contain questions/answers arrays.
    records = payload if isinstance(payload, list) else payload.get('data') if isinstance(payload, dict) else None
    if isinstance(records, list) and records and isinstance(records[0], dict) and 'questions' in records[0]:
        df = load_logic_dataset(path)
        examples = []
        for _, row in df.iterrows():
            examples.append({
                'id': row['id'],
                'group_id': row.get('group_id'),
                'question': row['question'],
                'premises-NL': list(row['premises_nl']),
                'gold_answer': str(row.get('gold_answer', '')).strip(),
                'question_type_hint': row.get('question_type'),
            })
        return examples

    # Kaggle inference style: either top-level instances or a list of flat request objects.
    if isinstance(payload, dict) and isinstance(payload.get('instances'), list):
        flat = payload['instances']
    elif isinstance(payload, list):
        flat = payload
    else:
        raise ValueError(f'Unsupported input shape in {path}')

    examples = []
    for index, item in enumerate(flat):
        example = dict(item)
        example.setdefault('id', f'logic_{index:04d}')
        answer = example.get('answer') or example.get('gold_answer') or example.get('label')
        if answer is not None:
            example['gold_answer'] = str(answer).strip()
        examples.append(example)
    return examples


examples = load_type1_examples(INPUT_PATH)
if LIMIT is not None:
    examples = examples[:LIMIT]

print('examples =', len(examples))
print('first keys =', sorted(examples[0].keys()))
print(json.dumps({k: examples[0].get(k) for k in ['id', 'question', 'gold_answer']}, ensure_ascii=False, indent=2)[:1200])


In [ ]:
# Build a Transformers-backed JSON LLM client through the project codebase.
# This intentionally avoids OpenAI-compatible server mode and loads the model in-process.
settings = Settings(
    llm_provider='local',
    llm_base_url=None,
    llm_model=MODEL_NAME,
    llm_api_key=None,
    mock_llm=False,
    llm_max_tokens=MAX_NEW_TOKENS,
    llm_temperature=LLM_TEMPERATURE,
    llm_top_p=LLM_TOP_P,
    type1_translation_samples=TYPE1_TRANSLATION_SAMPLES,
    type1_sampling_temperature=TYPE1_SAMPLING_TEMPERATURE,
)
translator_client = build_json_client_from_settings(settings)
assert translator_client is not None, 'LLM-only evaluation requires a Transformers JSON client.'
if CLEAR_KB_CACHE:
    clear_kb_cache()
    print('cleared Type 1 KB cache')
print('translator_client =', type(translator_client).__name__)
print('settings =', settings.model_dump(mode='json', exclude={'llm_api_key'}))


In [ ]:
def normalize_answer(value: object) -> str:
    return str(value or '').strip()


def compact_text(value: object, limit: int = 180) -> str:
    text = ' '.join(str(value or '').split())
    return text if len(text) <= limit else text[: limit - 3] + '...'


def should_log_case(index: int, total: int) -> bool:
    return bool(LOG_EVERY) and (index == 1 or index == total or index % LOG_EVERY == 0)


def trace_lines(prediction: dict, prefix: str) -> list[str]:
    return [str(line) for line in prediction.get('cot') or [] if str(line).startswith(prefix)]


def make_error_prediction(example: dict, route_reason: str, error: BaseException) -> dict:
    return {
        'id': example.get('id'),
        'task_type': 'type1_logic',
        'question_type': None,
        'answer': '',
        'explanation': f'Pipeline failed: {error}',
        'fol': None,
        'cot': [],
        'premises': [],
        'confidence': 0.0,
        'error': repr(error),
        'route_reason': route_reason,
        'official': {
            'answer': '',
            'explanation': f'Pipeline failed: {error}',
            'fol': None,
            'cot': [],
            'premises': [],
            'confidence': 0.0,
        },
        'traceback': traceback.format_exc(),
    }


router = TaskRouter()
predictions: list[dict] = []
started_at = time.monotonic()

for index, example in enumerate(examples, start=1):
    route_reason = ''
    try:
        request = PredictionRequest.model_validate(example)
        route = router.route(request)
        route_reason = route.reason
        if should_log_case(index, len(examples)):
            print(
                f"[{index}/{len(examples)}] id={request.id} route={route.task_type.value} "
                f"question_type={route.question_type.value} reason={route.reason}"
            )
            print('  question:', compact_text(request.question))
            print('  premises:', len(request.premises_nl or []))
        if route.task_type != TaskType.TYPE1_LOGIC:
            raise ValueError(f'Expected Type 1 logic request, got {route.task_type}')

        response = run_type1_pipeline(
            request,
            translator_client=translator_client,
            settings=settings,
            question_type=route.question_type,
        )
        prediction = response.model_dump(mode='json')
        prediction['route_reason'] = route.reason
        prediction['official'] = to_official_response(response)
        if should_log_case(index, len(examples)):
            print(
                f"  answer={prediction.get('answer')} "
                f"confidence={prediction.get('confidence')} error={bool(prediction.get('error'))}"
            )
            for line in trace_lines(prediction, 'symbolic_consistency_vote:'):
                print(' ', line)
            for line in trace_lines(prediction, 'cot_fallback_after_symbolic_unknown:'):
                print(' ', line)
            if prediction.get('error'):
                print('  pipeline_error:', compact_text(prediction.get('error'), 240))
    except Exception as exc:
        if not CONTINUE_ON_ERROR:
            raise
        prediction = make_error_prediction(example, route_reason, exc)
        if should_log_case(index, len(examples)):
            print(f"[{index}/{len(examples)}] id={example.get('id')} FAILED")
            print('  error:', repr(exc))
            if LOG_TRACEBACK:
                print(traceback.format_exc())

    if 'gold_answer' in example:
        prediction['gold_answer'] = normalize_answer(example.get('gold_answer'))
        prediction['is_correct'] = normalize_answer(prediction.get('answer')) == prediction['gold_answer']
        if should_log_case(index, len(examples)):
            print(
                f"  gold={prediction['gold_answer']} "
                f"correct={prediction['is_correct']}"
            )

    predictions.append(prediction)
    if PROGRESS_EVERY and (index % PROGRESS_EVERY == 0 or index == len(examples)):
        elapsed = time.monotonic() - started_at
        errors = sum(1 for item in predictions if item.get('error'))
        cot_fallbacks = sum(bool(trace_lines(item, 'cot_fallback_after_symbolic_unknown:')) for item in predictions)
        print(
            f'processed {index}/{len(examples)} | errors={errors} '
            f'| cot_fallbacks={cot_fallbacks} | elapsed={elapsed:.1f}s'
        )

elapsed = time.monotonic() - started_at
print('done elapsed_seconds =', round(elapsed, 2))


In [ ]:
def build_report(predictions: list[dict]) -> dict:
    symbolic_vote_lines = [
        line
        for item in predictions
        for line in trace_lines(item, 'symbolic_consistency_vote:')
    ]
    cot_fallback_lines = [
        line
        for item in predictions
        for line in trace_lines(item, 'cot_fallback_after_symbolic_unknown:')
    ]
    report = {
        'source': str(INPUT_PATH),
        'model': MODEL_NAME,
        'max_new_tokens': MAX_NEW_TOKENS,
        'llm_temperature': LLM_TEMPERATURE,
        'llm_top_p': LLM_TOP_P,
        'type1_translation_samples': TYPE1_TRANSLATION_SAMPLES,
        'type1_sampling_temperature': TYPE1_SAMPLING_TEMPERATURE,
        'count': len(predictions),
        'errors': sum(1 for item in predictions if item.get('error')),
        'cot_fallbacks': len(cot_fallback_lines),
        'symbolic_votes': len(symbolic_vote_lines),
        'answer_counts': dict(Counter(normalize_answer(item.get('answer')) for item in predictions)),
        'question_type_counts': dict(Counter(str(item.get('question_type')) for item in predictions)),
        'error_kinds': dict(Counter(compact_text(item.get('error'), 120) for item in predictions if item.get('error'))),
    }
    labeled = [item for item in predictions if 'gold_answer' in item]
    if labeled:
        correct = sum(1 for item in labeled if item.get('is_correct'))
        report['labeled_count'] = len(labeled)
        report['accuracy'] = correct / len(labeled)
        by_type: dict[str, list[int]] = defaultdict(lambda: [0, 0])
        for item in labeled:
            key = str(item.get('question_type'))
            by_type[key][0] += int(bool(item.get('is_correct')))
            by_type[key][1] += 1
        report['accuracy_by_question_type'] = {
            key: {'correct': value[0], 'total': value[1], 'accuracy': value[0] / value[1]}
            for key, value in sorted(by_type.items())
        }
    return report


report = build_report(predictions)
pd.DataFrame([report])


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

output_payload = {
    'source': str(INPUT_PATH),
    'format': 'exact_type1_llm_pipeline_eval',
    'count': len(predictions),
    'predictions': predictions,
}
OUTPUT_PATH.write_text(json.dumps(output_payload, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
REPORT_PATH.write_text(json.dumps(report, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')

print('wrote predictions:', OUTPUT_PATH)
print('wrote report     :', REPORT_PATH)
print(json.dumps(report, ensure_ascii=False, indent=2))


In [ ]:
# Inspect failed cases without changing pipeline behavior.
failed = [item for item in predictions if item.get('error')]
print('failed_count =', len(failed))
if failed:
    failed_df = pd.DataFrame(failed[:10])
    display_cols = [col for col in ['id', 'question_type', 'answer', 'gold_answer', 'error'] if col in failed_df.columns]
    failed_df[display_cols]
else:
    pd.DataFrame()
